# 01 — Generate Dirty Baseball Data

Build a synthetic **at-bat / pitch** dataset with intentional quality issues so DQX has
something to find. Roughly `DIRTY_FRACTION` (default 12%) of rows get one of:

| Issue | Field affected |
|---|---|
| Null `pitcher_id` or `batter_id` | identifiers |
| Pitch speed negative or > 130 mph | `pitch_speed_mph` |
| Inning > 20 | `inning` |
| Batting avg outside [0, 1] | `batting_avg` |
| Bogus pitch type code (e.g. `XX`) | `pitch_type` |
| Future game date | `game_date` |
| Duplicate `at_bat_id` | uniqueness |

Output: `{UC_SCHEMA}_bronze.pitches_raw` — Delta, no DQ enforcement yet.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
from datetime import date, timedelta
from faker import Faker
import os, random

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG     = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA      = os.getenv("UC_SCHEMA",  "dqx_baseball")
N_PITCHES      = int(os.getenv("N_PITCHES", "50000"))
DIRTY_FRACTION = float(os.getenv("DIRTY_FRACTION", "0.12"))
SEED           = int(os.getenv("RANDOM_SEED", "42"))

BRONZE = f"{UC_CATALOG}.{UC_SCHEMA}_bronze"
TABLE  = f"{BRONZE}.pitches_raw"

random.seed(SEED)
Faker.seed(SEED)
fake = Faker()

print(f"Generating {N_PITCHES:,} pitches → {TABLE}")
print(f"Dirty fraction: {DIRTY_FRACTION:.0%}")

Generating 50,000 pitches → alexander_booth.dqx_baseball_bronze.pitches_raw
Dirty fraction: 12%


In [2]:
# Reference data — small fixed pools so we can later check uniqueness / membership
PITCH_TYPES   = ["FF", "SI", "SL", "CU", "CH", "FC", "KC", "FS"]
BAD_TYPES     = ["XX", "ZZ", "??", ""]  # values DQX should flag
PITCHER_POOL  = [f"P{1000 + i}" for i in range(120)]
BATTER_POOL   = [f"B{2000 + i}" for i in range(240)]
GAME_POOL     = [f"G{20260000 + i}" for i in range(800)]
TODAY         = date(2026, 5, 8)
SEASON_START  = date(2026, 3, 28)

def random_game_date():
    days = random.randint(0, (TODAY - SEASON_START).days)
    return SEASON_START + timedelta(days=days)

def clean_row(i):
    return {
        "at_bat_id":       f"AB{i:08d}",
        "game_id":         random.choice(GAME_POOL),
        "game_date":       random_game_date(),
        "inning":          random.randint(1, 9) if random.random() < 0.92 else random.randint(10, 14),
        "pitcher_id":      random.choice(PITCHER_POOL),
        "batter_id":       random.choice(BATTER_POOL),
        "pitch_type":      random.choice(PITCH_TYPES),
        "pitch_speed_mph": round(random.uniform(72.0, 102.0), 1),
        "batting_avg":     round(random.uniform(0.180, 0.380), 3),
    }

DIRTY_MUTATORS = [
    lambda r: r.update({"pitcher_id": None}),
    lambda r: r.update({"batter_id": None}),
    lambda r: r.update({"pitch_speed_mph": -1 * r["pitch_speed_mph"]}),
    lambda r: r.update({"pitch_speed_mph": round(random.uniform(115.0, 140.0), 1)}),
    lambda r: r.update({"inning": random.randint(25, 99)}),
    lambda r: r.update({"batting_avg": round(random.uniform(1.01, 2.5), 3)}),
    lambda r: r.update({"batting_avg": round(random.uniform(-0.5, -0.01), 3)}),
    lambda r: r.update({"pitch_type": random.choice(BAD_TYPES)}),
    lambda r: r.update({"game_date": TODAY + timedelta(days=random.randint(30, 365))}),
]

def maybe_dirty(row):
    if random.random() < DIRTY_FRACTION:
        random.choice(DIRTY_MUTATORS)(row)
    return row

rows = [maybe_dirty(clean_row(i)) for i in range(N_PITCHES)]

# Inject a few duplicate at_bat_ids so the is_unique check has work to do
n_dupes = max(5, N_PITCHES // 1000)
for _ in range(n_dupes):
    src, dst = random.sample(range(N_PITCHES), 2)
    rows[dst]["at_bat_id"] = rows[src]["at_bat_id"]

print(f"Built {len(rows):,} rows ({n_dupes} duplicate at_bat_ids planted)")

Built 50,000 rows (50 duplicate at_bat_ids planted)


In [3]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)

schema = StructType([
    StructField("at_bat_id",       StringType(),  True),
    StructField("game_id",         StringType(),  True),
    StructField("game_date",       DateType(),    True),
    StructField("inning",          IntegerType(), True),
    StructField("pitcher_id",      StringType(),  True),
    StructField("batter_id",       StringType(),  True),
    StructField("pitch_type",      StringType(),  True),
    StructField("pitch_speed_mph", DoubleType(),  True),
    StructField("batting_avg",     DoubleType(),  True),
])

df = spark.createDataFrame(rows, schema=schema)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE)

print(f"Wrote {df.count():,} rows to {TABLE}")

Wrote 50,000 rows to alexander_booth.dqx_baseball_bronze.pitches_raw


In [4]:
# Quick smoke check on what we planted — should see a non-trivial chunk of bad data
spark.sql(f"""
    SELECT
        COUNT(*)                                                  AS total,
        SUM(CASE WHEN pitcher_id IS NULL THEN 1 ELSE 0 END)        AS null_pitcher,
        SUM(CASE WHEN pitch_speed_mph < 40 OR pitch_speed_mph > 110 THEN 1 ELSE 0 END) AS bad_speed,
        SUM(CASE WHEN inning > 20 THEN 1 ELSE 0 END)               AS bad_inning,
        SUM(CASE WHEN batting_avg < 0 OR batting_avg > 1 THEN 1 ELSE 0 END) AS bad_avg,
        SUM(CASE WHEN pitch_type NOT IN ('FF','SI','SL','CU','CH','FC','KC','FS') THEN 1 ELSE 0 END) AS bad_type,
        SUM(CASE WHEN game_date > current_date() THEN 1 ELSE 0 END) AS future_date
    FROM {TABLE}
""").show(truncate=False)

+-----+------------+---------+----------+-------+--------+-----------+
|total|null_pitcher|bad_speed|bad_inning|bad_avg|bad_type|future_date|
+-----+------------+---------+----------+-------+--------+-----------+
|50000|641         |1295     |693       |1342   |671     |663        |
+-----+------------+---------+----------+-------+--------+-----------+



Bronze is loaded with a known mess. Continue with `02_profile_and_generate_rules`.